In [0]:
spark.sql("SHOW TABLES IN samples.wanderbricks").display

In [0]:
# Cell 2 — CDC shape, decides your AUTO CDC clause later
spark.read.table("samples.wanderbricks.booking_updates").limit(20).display()
spark.sql("DESCRIBE TABLE samples.wanderbricks.booking_updates").display()


In [0]:
# Validate the CDC assumptions before you build a pipeline on them
spark.sql("""
SELECT
  COUNT(*)                                   AS updates,
  COUNT(DISTINCT booking_id)                 AS keys,
  SUM(CASE WHEN updated_at IS NULL THEN 1 END) AS null_seq,
  MIN(updated_at), MAX(updated_at)
FROM samples.wanderbricks.booking_updates
""").display()

# Ties on (key, sequence) would break ordering — expect zero rows
spark.sql("""
SELECT booking_id, updated_at, COUNT(*) c
FROM samples.wanderbricks.booking_updates
GROUP BY 1,2 HAVING COUNT(*) > 1
""").display()

# Status domain — drives your metric view dimensions
spark.sql("SELECT status, COUNT(*) FROM samples.wanderbricks.booking_updates GROUP BY 1 ORDER BY 2 DESC").display()

In [0]:
%sql
SHOW CATALOGS

In [0]:
initials = "maxm"
print (initials)

In [0]:
# The catalog needs to be created 
CATALOG = 'ibm_dbx_sandbox'
SCHEMA = initials+'_bronze'
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
# Land a bronze copy you own, so Git/DAB/Lakeflow demos have writable target
# CATALOG, SCHEMA = 'ibm_dbx_sandbox', 'samples.wanderbricks'
for t in ["bookings", "booking_updates", "payments", "properties", "users"]:
    (spark.read.table(f"samples.wanderbricks.{t}")).write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{t}")

In [0]:
%sql
SELECT * FROM ibm_dbx_sandbox.information_schema.tables WHERE table_schema = 'maxm_bronze';

In [0]:
%sql
SHOW GRANTS ON SCHEMA ibm_dbx_sandbox.maxm_bronze

In [0]:
%sql
DESCRIBE TABLE EXTENDED ibm_dbx_sandbox.maxm_bronze.bookings;